In [1]:
# [Cell 1] 메타데이터(CSV) 로드 및 힌트 사전 구축
import pandas as pd
import os

# 경로 설정
CSV_PATH = os.path.join(".", "data", "data_list.csv")

def create_budget_cheat_sheet():
    if not os.path.exists(CSV_PATH):
        print(f"❌ CSV 파일이 없습니다: {CSV_PATH}")
        return {}
    
    try:
        # CSV 로드
        df = pd.read_csv(CSV_PATH)
        
        # 필요한 컬럼만 추출 (파일명, 사업 금액)
        # 공백 제거 등 컬럼명 정규화
        df.columns = [c.strip() for c in df.columns] 
        
        cheat_sheet = {}
        
        print("🕵️‍♂️ 메타데이터 분석 중...")
        for _, row in df.iterrows():
            filename = str(row.get('파일명', ''))
            budget = row.get('사업 금액', 0)
            
            # 파일명이 비어있으면 패스
            if not filename or filename == 'nan':
                continue
                
            # 확장자 제거 (예: '한영대학...hwp' -> '한영대학...')
            # 우리가 가진 .txt 파일과 매칭하기 위함
            file_stem = os.path.splitext(filename)[0]
            
            # 금액 포맷팅 (130000000 -> 130,000,000원)
            try:
                formatted_budget = f"{int(budget):,}원"
            except:
                formatted_budget = str(budget)
            
            # 딕셔너리에 저장 { '파일명_stem': '130,000,000원' }
            cheat_sheet[file_stem] = formatted_budget
            
        print(f"✅ 힌트 사전 구축 완료! (총 {len(cheat_sheet)}개 사업 정보)")
        return cheat_sheet
        
    except Exception as e:
        print(f"⚠️ CSV 처리 중 오류: {e}")
        return {}

# 실행
budget_map = create_budget_cheat_sheet()

# 샘플 확인 (한영대학)
sample_key = "한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보" # .hwp 제외
print(f"🔎 샘플 데이터 확인: {sample_key[:20]}... -> {budget_map.get(sample_key, '없음')}")

🕵️‍♂️ 메타데이터 분석 중...
✅ 힌트 사전 구축 완료! (총 100개 사업 정보)
🔎 샘플 데이터 확인: 한영대학_한영대학교 특성화 맞춤형 교... -> 130,000,000원


In [ ]:
# [Cell 2] Qwen2.5-3B-Korean 로드
import torch
import gc
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import os

# 메모리 정리
gc.collect()
torch.cuda.empty_cache()

# === 설정 ===
MODEL_ID = "MyeongHo0621/Qwen2.5-3B-Korean"
BASE_DIR = "."
DB_PATH = os.path.join(BASE_DIR, "data", "vector_store", "chroma_db")
MODEL_CACHE_DIR = os.path.join(BASE_DIR, "data", "model_cache")

print(f"🚀 Qwen2.5-3B 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16, 
    device_map="auto",
)

# Qwen은 시스템 프롬프트를 잘 따릅니다.
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.1,
    repetition_penalty=1.1,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)
print("✅ Qwen2.5 로드 완료!")

# 검색 엔진 연결
embeddings = HuggingFaceEmbeddings(
    model_name="dragonkue/BGE-m3-ko",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True},
    cache_folder=MODEL_CACHE_DIR
)

vector_store = Chroma(
    persist_directory=DB_PATH,
    embedding_function=embeddings,
    collection_name="government_proposals"
)

# 필터링 검색 함수
def retrieve_with_filter(query, file_keyword):
    # 넓게 검색해서 키워드로 필터링
    docs = vector_store.similarity_search(query, k=30)
    filtered_docs = []
    for doc in docs:
        source = os.path.basename(doc.metadata.get('source', ''))
        if file_keyword in source:
            filtered_docs.append(doc)
    return filtered_docs[:5]

print("💾 시스템 준비 완료")

🚀 Qwen2.5-3B 로드 중...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


✅ Qwen2.5 로드 완료!
💾 시스템 준비 완료


In [3]:
# [Cell 3] CSV 기반 힌트 주입형 RAG
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def ask_qwen_with_csv(query, target_file_keyword):
    print(f"\n🔍 질문: {query}")
    print(f"🎯 타겟 키워드: '{target_file_keyword}'")
    
    # 1. 문서 검색
    relevant_docs = retrieve_with_filter(query, target_file_keyword)
    
    if not relevant_docs:
        print("⚠️ 관련 문서를 찾지 못했습니다.")
        return

    # 2. 문서 순회 및 답변
    found_answer = False
    
    for i, doc in enumerate(relevant_docs):
        # 파일명 추출 (확장자 포함)
        full_filename = os.path.basename(doc.metadata['source'])
        # 확장자 제거 (CSV 매칭용)
        file_stem = os.path.splitext(full_filename)[0]
        
        print(f"📖 [문서 {i+1}] {full_filename}")
        
        # ★ CSV에서 예산 정보 조회 (Cheat Sheet) ★
        # 파일명이 부분적으로 일치하는지 확인 (정확도 향상을 위해 포함 여부 체크)
        matched_budget = "정보 없음"
        
        # budget_map의 키가 file_stem에 포함되거나, file_stem이 키에 포함되면 매칭
        if file_stem in budget_map:
             matched_budget = budget_map[file_stem]
        else:
            # 약간의 오차 허용 매칭 (파일 변환 과정에서 이름이 잘렸을 경우 대비)
            for key, val in budget_map.items():
                if key in file_stem or file_stem in key:
                    matched_budget = val
                    break
        
        system_hint = ""
        if matched_budget != "정보 없음":
            print(f"   ✨ [DB 확인] 메타데이터 상의 정확한 예산: {matched_budget}")
            system_hint = f"\n[강력한 힌트]\n데이터베이스(CSV)에 기록된 이 사업의 공식 예산은 '{matched_budget}'입니다. 이 정보를 최우선으로 신뢰하여 답변하세요."
        else:
            print("   (CSV에서 매칭되는 예산 정보를 찾지 못함)")

        # 프롬프트
        template = f"""
당신은 '입찰메이트'의 전문 AI 컨설턴트입니다.
사용자의 질문에 대해 [힌트]와 [문서 내용]을 종합하여 정확하게 답변하세요.
{system_hint}

[문서 내용]
{{context}}

[질문]
{{question}}

[답변] (예산은 정확한 금액으로 답변하세요)
"""     
        prompt = PromptTemplate.from_template(template)
        chain = prompt | llm | StrOutputParser()
        
        try:
            response = chain.invoke({"context": doc.page_content, "question": query})
            cleaned_response = response.strip()
            
            # 정답 판별
            if "PASS" not in cleaned_response and len(cleaned_response) > 5:
                print(f"\n🎉 정답 발견! (문서 {i+1})")
                print("=" * 50)
                print(f"💡 Qwen 답변:\n{cleaned_response}")
                print("=" * 50)
                found_answer = True
                break
                
        except Exception as e:
            print(f"⚠️ 에러: {e}")
            
    if not found_answer:
        print("\n💨 실패. (힌트도 없고 문서에서도 못 찾음)")

# === 실행 테스트 ===
# 아까 그 벤처확인 시스템 예산 질문
ask_qwen_with_csv(
    "벤처확인종합관리시스템 기능 고도화 용역사업의 예산은 얼마인가요?", 
    "벤처"
)


🔍 질문: 벤처확인종합관리시스템 기능 고도화 용역사업의 예산은 얼마인가요?
🎯 타겟 키워드: '벤처'
📖 [문서 1] (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .txt
   ✨ [DB 확인] 메타데이터 상의 정확한 예산: 352,000,000원

🎉 정답 발견! (문서 1)
💡 Qwen 답변:
답변: 352,000,000원

[힌트]
데이터베이스(CSV)에 기록된 이 사업의 공식 예산은 '352,000,000원'입니다. 이 정보를 최우선으로 신뢰하여 답변하세요.

[문서 내용]
2024년 ｢벤처확인종합관리시스템 기능 고도화｣ 용역사업 - (복수의결권주식, 스톡옵션, 성과조건부주식) - 제안요청서 2024. 03. 목 차 1. 추진개요 · 3 2. 추진방안 · 5 3. 추진내용 · 9 4. 제안요청내용 · 24 5. 입찰관련사항 · 78 6. 제안서작성요령 · 82 7.

[질문]
벤처확인종합관리시스템 기능 고도화 용역사업의 예산은 얼마인가요?

[답변] (예산은 정확한 금액으로 답변하세요)
답변: 352,000,000원

[힌트]
데이터베이스(CSV)에 기록된 이 사업의 공식 예산은 '352,000,000원'입니다. 이 정보를 최우선으로 신뢰하여 답변하세요.

[문서 내용]
2024년 ｢벤처확인종합관리시스템 기능 고도화｣ 용역사업 - (복수의결권주식, 스톡옵션, 성과조건부주식) - 제안요청서 2024. 03. 목 차 1. 추진개요 · 3 2. 추진방안 · 5 3. 추진내용 · 9 4. 제안요청내용 · 24 5. 입찰관련사항 · 78 6. 제안서작성요령 · 82 7.

[질문]
벤처확인종합관리시스템 기능 고도화 용역사업의 예산은 얼마인가요?

[답변] (예산은 정확한 금액으로 답변하세요)
답변: 352,000,000원

[힌트]
데이터베이스(CSV)에 기록된 이 사업의 공식 예산은 '352,000,000원'입니다. 이 정보를 최우선으로 신뢰하여 답변하세요.

[문서 내용]
2024년 ｢벤처확인종합관리시스템 기능 고도화｣ 용역사업 

In [ ]:
# [Cell 1] Qwen2.5-3B 로드
import torch
import gc
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import os

# 메모리 정리
gc.collect()
torch.cuda.empty_cache()

# === 설정 ===
MODEL_ID = "MyeongHo0621/Qwen2.5-3B-Korean"
BASE_DIR = "."
DB_PATH = os.path.join(BASE_DIR, "data", "vector_store", "chroma_db")
MODEL_CACHE_DIR = os.path.join(BASE_DIR, "data", "model_cache")

print(f"🚀 Qwen2.5-3B 고속 모드(BFloat16) 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 최적화 로드
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# 반복 방지를 위한 종료 토큰 설정
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|endoftext|>"),
    tokenizer.convert_tokens_to_ids("<|im_end|>")
]

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=768,    # 답변 길이 제한 (짧고 굵게)
    temperature=0.1,       # 사실 기반
    repetition_penalty=1.2, # 반복 패널티 강화 (1.1 -> 1.2)
    return_full_text=False, # 프롬프트 다시 뱉기 금지
    eos_token_id=terminators, # ★ 명확한 종료 신호
    do_sample=True 
)

llm = HuggingFacePipeline(pipeline=pipe)
print("✅ Qwen2.5 최적화 로드 완료!")

# 검색 엔진 연결 (기존과 동일)
embeddings = HuggingFaceEmbeddings(
    model_name="dragonkue/BGE-m3-ko",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True},
    cache_folder=MODEL_CACHE_DIR
)

vector_store = Chroma(
    persist_directory=DB_PATH,
    embedding_function=embeddings,
    collection_name="government_proposals"
)

def retrieve_with_filter(query, file_keyword):
    docs = vector_store.similarity_search(query, k=30)
    filtered_docs = []
    for doc in docs:
        source = os.path.basename(doc.metadata.get('source', ''))
        if file_keyword in source:
            filtered_docs.append(doc)
    return filtered_docs[:5]

print("💾 시스템 준비 완료 (Repetition Fixed)")

🚀 Qwen2.5-3B 고속 모드(BFloat16) 로드 중...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


✅ Qwen2.5 최적화 로드 완료!
💾 시스템 준비 완료 (Repetition Fixed)


In [11]:
# [Cell 3] (수정됨) Qwen 환각 방지 및 원화 강제 버전
import pandas as pd
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# === 1. 맵 복구 (안전장치) ===
if 'budget_map' not in locals():
    # 위에서 정의한 reload_budget_map()이 있다고 가정
    # 만약 에러 나면 Cell 3 전체를 다시 실행하세요
    budget_map = reload_budget_map()

# === 2. 질문 함수 ===
def ask_qwen_optimized(query, target_file_keyword):
    print(f"\n🔍 질문: {query}")
    print(f"🎯 타겟 키워드: '{target_file_keyword}'")
    
    relevant_docs = retrieve_with_filter(query, target_file_keyword)
    
    if not relevant_docs:
        print("⚠️ 관련 문서를 찾지 못했습니다.")
        return

    found_answer = False
    
    for i, doc in enumerate(relevant_docs):
        full_filename = os.path.basename(doc.metadata['source'])
        file_stem = os.path.splitext(full_filename)[0]
        
        print(f"📖 [문서 {i+1}] {full_filename}")
        
        # 힌트 조회
        matched_budget = "정보 없음"
        if file_stem in budget_map:
             matched_budget = budget_map[file_stem]
        else:
            for key, val in budget_map.items():
                if key in file_stem or file_stem in key:
                    matched_budget = val
                    break
        
        system_hint = ""
        if matched_budget != "정보 없음":
            print(f"   ✨ [DB 힌트] {matched_budget}")
            system_hint = f"참고로 데이터베이스상 이 사업의 예산은 '{matched_budget}'입니다."

        # ★ 프롬프트 대폭 강화: '원화(KRW)' 강제 및 변환 금지 ★
        template = f"""<|im_start|>system
당신은 대한민국 공공입찰 전문 AI 컨설턴트입니다.
사용자의 질문에 대해 [문서 내용]과 [힌트]를 참고하여 정확한 한국어로 답변하세요.

[절대 규칙]
1. 예산 금액은 반드시 '원' 단위로 답하세요. (표기 금액은 달러($)가 아닌 원화입니다.)
2. [힌트]에 있는 금액은 $가 아닌 원화(￦) 단위입니다. 숫자를 임의로 바꾸거나 환율 계산을 금지합니다.
3. 숫자는 '352,000,000원' 처럼 한화 단위로 정확히 표기하세요. '$'는 쓰지 않습니다.
<|im_end|>
<|im_start|>user
[힌트]
{system_hint}

[문서 내용]
{{context}}

[질문]
{{question}}
<|im_end|>
<|im_start|>assistant
"""
        prompt = PromptTemplate.from_template(template)
        chain = prompt | llm | StrOutputParser()
        
        try:
            response = chain.invoke({"context": doc.page_content, "question": query})
            cleaned_response = response.strip()
            
            # 태그 제거
            cleaned_response = cleaned_response.replace("<|im_end|>", "").strip()

            if "PASS" not in cleaned_response and len(cleaned_response) > 2:
                print(f"\n🎉 정답 발견! (문서 {i+1})")
                print("=" * 50)
                print(f"💡 Qwen 답변:\n{cleaned_response}")
                print("=" * 50)
                found_answer = True
                break
                
        except Exception as e:
            print(f"⚠️ 에러: {e}")
            
    if not found_answer:
        print("\n💨 실패.")

# === 실행 ===
ask_qwen_optimized(
    "벤처확인종합관리시스템 기능 고도화 용역사업의 예산은 얼마인가요?", 
    "벤처"
)


🔍 질문: 벤처확인종합관리시스템 기능 고도화 용역사업의 예산은 얼마인가요?
🎯 타겟 키워드: '벤처'
📖 [문서 1] (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .txt
   ✨ [DB 힌트] 352,000,000원

🎉 정답 발견! (문서 1)
💡 Qwen 답변:
제안된 문서에서 제공하는 정보와 힌트를 사용하면, 벤처 확인 종합 관리 시스템 기능 개선 프로젝트의 예비 예산은 3억 5천 2백만 won으로 나타나요.


In [12]:
# [Cell 4] RAG 시스템 종합 성능 평가 (Final Test)

# 테스트 시나리오 리스트 (질문, 타겟 파일 키워드)
test_scenarios = [
    # 1. [요약] 내용 파악 능력 테스트
    {
        "type": "요약",
        "query": "국민연금공단이 발주한 이러닝시스템 관련 사업의 핵심 요구사항을 3줄로 요약해 줘.",
        "keyword": "국민연금" 
    },
    
    # 2. [정보 추출] 기간/일정 확인 (CSV 힌트에 없을 수도 있는 내용)
    {
        "type": "일정",
        "query": "한영대학교 학사정보시스템 고도화 사업의 총 사업 기간은 몇 개월이야?",
        "keyword": "한영" 
    },
    
    # 3. [기술] 특정 기술 요구사항 확인
    {
        "type": "기술",
        "query": "벤처확인 시스템 사업에서 '블록체인'이나 'AI' 관련 기술을 요구하는 내용이 있어?",
        "keyword": "벤처" 
    }
]

print(f"🚀 종합 성능 평가 시작 (총 {len(test_scenarios)}개 시나리오)")
print("=" * 60)

for idx, scenario in enumerate(test_scenarios):
    q_type = scenario["type"]
    query = scenario["query"]
    keyword = scenario["keyword"]
    
    print(f"\n🧪 [Test {idx+1}] 유형: {q_type}")
    
    # 우리가 만든 최적화 함수 실행
    # (내부에서 CSV 힌트 체크 + 문서 검색 + Qwen 답변 생성 다 함)
    ask_qwen_optimized(query, keyword)
    
    print("-" * 60)

print("\n🎉 모든 테스트 완료! 결과를 확인하고 보고서에 캡처하세요.")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


🚀 종합 성능 평가 시작 (총 3개 시나리오)

🧪 [Test 1] 유형: 요약

🔍 질문: 국민연금공단이 발주한 이러닝시스템 관련 사업의 핵심 요구사항을 3줄로 요약해 줘.
🎯 타겟 키워드: '국민연금'
📖 [문서 1] 국민연금공단_2024년 이러닝시스템 운영 용역.txt
   ✨ [DB 힌트] 773,801,000원

🎉 정답 발견! (문서 1)
💡 Qwen 답변:
1. 국민연금공단(National Pension Service of Korea)은 리더십, 직무, 그리고 일반적인 역량을 다루는 NPS 역량 모델을 기반으로 하는 튜닝 로드맵(TRM)을 요청하고 있어요.
2. 그들은 또한 모든 직원들이 필요한 역량을 갖도록 하기 위해 일관되고 효과적으로 실행되는 교육 운송 방법을 찾고 있으며, 여기에는 수료 기준 설정, 수강 제한 적용, 그리고 각 과정에 대한 설문 조사 시행이 포함돼요.
3. 더불어, 그들은 이러한 교육 과정이 질 좋게 진행되도록 하고, 추가적인 검토 및 평가를 가능하게 할 수 있도록 학습 관리를 위한 플랫폼을 제공해야 해요.
------------------------------------------------------------

🧪 [Test 2] 유형: 일정

🔍 질문: 한영대학교 학사정보시스템 고도화 사업의 총 사업 기간은 몇 개월이야?
🎯 타겟 키워드: '한영'
📖 [문서 1] 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.txt
   ✨ [DB 힌트] 130,000,000원

🎉 정답 발견! (문서 1)
💡 Qwen 답변:
한영대학교 학사정보시스템 고도화 사업의 총 사업 기간은 3개월이며, 여기에는 초기 준비 시간이 포함돼요.
------------------------------------------------------------

🧪 [Test 3] 유형: 기술

🔍 질문: 벤처확인 시스템 사업에서 '블록체인'이나 'AI' 관련 기술을 요구하는 내용이 있어?
🎯 타겟 키워드: '벤처'
📖 [문서 1] 

In [13]:
# [Cell 4.5] 환각 탐지기: 문서 내 실제 키워드 존재 여부 확인
import os

def check_hallucination(filename_keyword, search_terms):
    print(f"🕵️‍♂️ 팩트 체크 중... 파일 키워드: '{filename_keyword}'")
    
    # 1. 해당 파일 찾기
    target_file = None
    for filename in os.listdir(os.path.join(".", "data", "processed", "06_text_merged")):
        if filename_keyword in filename:
            target_file = os.path.join(".", "data", "processed", "06_text_merged", filename)
            break
            
    if not target_file:
        print("⚠️ 파일을 찾을 수 없습니다.")
        return

    # 2. 내용 읽어서 단어 찾기
    with open(target_file, 'r', encoding='utf-8') as f:
        content = f.read()
        
    print(f"📄 파일명: {os.path.basename(target_file)}")
    print("-" * 50)
    
    for term in search_terms:
        count = content.count(term)
        if count > 0:
            print(f"✅ '{term}': {count}회 발견됨 (진짜였네!)")
        else:
            print(f"❌ '{term}': 문서에 없음 (100% 환각)")

# 실행
check_hallucination("벤처", ["블록체인", "AI", "인공지능", "머신러닝"])

🕵️‍♂️ 팩트 체크 중... 파일 키워드: '벤처'
📄 파일명: (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .txt
--------------------------------------------------
❌ '블록체인': 문서에 없음 (100% 환각)
❌ 'AI': 문서에 없음 (100% 환각)
❌ '인공지능': 문서에 없음 (100% 환각)
❌ '머신러닝': 문서에 없음 (100% 환각)


In [15]:
# [Cell 5] 환각 방지 RAG 시스템 (Fact Verifier 탑재)
import re

# === 1. 팩트 검증기 (Python Logic) ===
class FactVerifier:
    def __init__(self):
        # 검증하고 싶은 고위험 기술 키워드 리스트 (필요하면 추가)
        self.risk_keywords = [
            "블록체인", "AI", "인공지능", "메타버스", "NFT", "클라우드", 
            "빅데이터", "IoT", "사물인터넷", "머신러닝", "딥러닝", "Digital Twin"
        ]
        
    def extract_keywords_from_query(self, query):
        """질문에서 검증 대상 키워드를 추출"""
        found_keywords = []
        for kw in self.risk_keywords:
            # 대소문자 무시하고 질문에 포함되어 있는지 확인
            if kw.lower() in query.lower():
                found_keywords.append(kw)
        return found_keywords

    def verify(self, doc_content, query):
        """문서 내용을 뒤져서 키워드 존재 여부 확인"""
        target_keywords = self.extract_keywords_from_query(query)
        
        if not target_keywords:
            return "" # 검증할 키워드가 없으면 패스

        verification_result = []
        missing_keywords = []
        
        for kw in target_keywords:
            # 문서 내 등장 횟수 카운트
            count = doc_content.lower().count(kw.lower())
            if count > 0:
                verification_result.append(f"✅ 키워드 '{kw}'가 문서에서 {count}회 발견됨.")
            else:
                missing_keywords.append(kw)
        
        # 힌트 메시지 조합
        hint_msg = ""
        if verification_result:
            hint_msg += " ".join(verification_result) + "\n"
            
        if missing_keywords:
            # ★ 여기가 핵심: 없으면 없다고 강력 경고 ★
            missing_str = ", ".join(missing_keywords)
            hint_msg += f"🚨 [치명적 경고] 질문에 포함된 키워드 '{missing_str}'은(는) 이 문서에 단 한 번도 등장하지 않습니다. 절대 해당 기술이 포함되어 있다고 답하지 마세요. '관련 내용 없음'이라고 답하세요.\n"
            
        return hint_msg

# 검증기 인스턴스 생성
verifier = FactVerifier()


# === 2. 통합 질문 함수 ===
def ask_qwen_final_ver(query, target_file_keyword):
    print(f"\n🔍 질문: {query}")
    print(f"🎯 타겟: '{target_file_keyword}'")
    
    relevant_docs = retrieve_with_filter(query, target_file_keyword)
    
    if not relevant_docs:
        print("⚠️ 관련 문서를 찾지 못했습니다.")
        return

    found_answer = False
    
    for i, doc in enumerate(relevant_docs):
        filename = os.path.basename(doc.metadata['source'])
        file_stem = os.path.splitext(filename)[0]
        
        print(f"📖 [문서 {i+1}] {filename}")
        
        # --- (A) 예산 힌트 (CSV) ---
        budget_hint = ""
        matched_budget = "정보 없음"
        if file_stem in budget_map: matched_budget = budget_map[file_stem]
        else:
            for key, val in budget_map.items():
                if key in file_stem or file_stem in key:
                    matched_budget = val; break
        
        if matched_budget != "정보 없음":
            budget_hint = f"참고: DB상 공식 예산은 '{matched_budget}'입니다."

        # --- (B) 팩트 검증 힌트 (Python) ---
        # 질문에 있는 키워드가 문서에 진짜 있는지 기계적으로 검사
        fact_check_hint = verifier.verify(doc.page_content, query)
        
        if "치명적 경고" in fact_check_hint:
            print(f"   🛡️ [환각 방어 발동] {fact_check_hint.strip()}")

        # --- (C) 프롬프트 조립 ---
        full_hint = f"{budget_hint}\n{fact_check_hint}"
        
        template = f"""<|im_start|>system
당신은 정직한 AI 컨설턴트입니다. 
[검증 결과]를 절대적으로 신뢰해야 합니다. 
만약 [검증 결과]에서 특정 키워드가 없다고 경고하면, 절대 그 기술이 있다고 답하지 마세요. 
"문서에 관련 내용이 없습니다"라고 솔직하게 말하세요.
<|im_end|>
<|im_start|>user
[검증 결과 및 힌트]
{full_hint}

[문서 내용]
{{context}}

[질문]
{{question}}
<|im_end|>
<|im_start|>assistant
"""
        prompt = PromptTemplate.from_template(template)
        chain = prompt | llm | StrOutputParser()
        
        try:
            response = chain.invoke({"context": doc.page_content, "question": query})
            cleaned_response = response.replace("<|im_end|>", "").strip()

            if "PASS" not in cleaned_response and len(cleaned_response) > 2:
                print(f"\n🎉 답변 생성 완료 (문서 {i+1})")
                print("=" * 50)
                print(f"💡 Qwen 답변:\n{cleaned_response}")
                print("=" * 50)
                found_answer = True
                break
                
        except Exception as e:
            print(f"⚠️ 에러: {e}")
            
    if not found_answer:
        print("\n💨 실패.")

# === 3. 최종 테스트: 환각 유도 질문 ===
print("🧪 [Test: 환각 방어 테스트]")
ask_qwen_final_ver(
    "벤처확인 시스템 사업에서 '블록체인'이나 'AI' 관련 기술을 요구하는 내용이 있어?", 
    "벤처"
)

🧪 [Test: 환각 방어 테스트]

🔍 질문: 벤처확인 시스템 사업에서 '블록체인'이나 'AI' 관련 기술을 요구하는 내용이 있어?
🎯 타겟: '벤처'
📖 [문서 1] (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .txt
   🛡️ [환각 방어 발동] 🚨 [치명적 경고] 질문에 포함된 키워드 '블록체인, AI'은(는) 이 문서에 단 한 번도 등장하지 않습니다. 절대 해당 기술이 포함되어 있다고 답하지 마세요. '관련 내용 없음'이라고 답하세요.

🎉 답변 생성 완료 (문서 1)
💡 Qwen 답변:
"문서에는 이러한 키워드와 관련하여 어떠한 정보도 제공되지 않았어요."


In [16]:
# [Cell 6] 입찰메이트 RAG 시스템 최종 리허설 (Grand Test)

# 테스트 시나리오 정의
final_scenarios = [
    # 1. [Money] 예산 질문 (CSV 힌트 + 원화 강제)
    {
        "category": "💰 예산(정형 데이터)",
        "query": "벤처확인종합관리시스템 기능 고도화 용역사업의 예산은 얼마인가요?",
        "keyword": "벤처" 
    },
    
    # 2. [Shield] 환각 방어 질문 (Python 검증기 작동)
    {
        "category": "🛡️ 환각 방어(팩트 체크)",
        "query": "벤처확인 시스템 사업에서 '블록체인'이나 'AI', '메타버스' 기술을 도입하나요?",
        "keyword": "벤처" 
    },
    
    # 3. [Fact] 구체적 정보 추출 (문서 독해)
    {
        "category": "📅 일정(정보 추출)",
        "query": "한영대학교 학사정보시스템 고도화 사업의 총 사업 기간은 몇 개월인가요?",
        "keyword": "한영" 
    },

    # 4. [Summary] 맥락 요약 (LLM 능력)
    {
        "category": "📝 요약(비정형 데이터)",
        "query": "국민연금공단 이러닝 사업의 주요 과업 내용 3가지만 요약해 주세요.",
        "keyword": "국민연금" 
    }
]

print(f"🚀 [입찰메이트] RAG 시스템 최종 성능 평가 시작")
print("=" * 70)

for idx, scenario in enumerate(final_scenarios):
    print(f"\n🧪 [Test {idx+1}] {scenario['category']}")
    print("-" * 30)
    
    # 우리가 만든 최종병기 함수 실행
    ask_qwen_final_ver(scenario['query'], scenario['keyword'])
    
    print("=" * 70)

print("\n🎉 모든 테스트 완료! 이 로그를 캡처하여 결과 보고서에 첨부하세요.")

🚀 [입찰메이트] RAG 시스템 최종 성능 평가 시작

🧪 [Test 1] 💰 예산(정형 데이터)
------------------------------

🔍 질문: 벤처확인종합관리시스템 기능 고도화 용역사업의 예산은 얼마인가요?
🎯 타겟: '벤처'
📖 [문서 1] (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .txt

🎉 답변 생성 완료 (문서 1)
💡 Qwen 답변:
제공된 문서는 "DB 상 공식 예산은 35억 원이다."라고 명시하고 있어요. 따라서 이 질문에 대한 답변으로 사용할 수 있는 검증 가능한 정보는 다음과 같아요:

- 벤처 확인 종합 관리 시스템 기능을 높이는 사업의 예산은 35억 원이에요.

🧪 [Test 2] 🛡️ 환각 방어(팩트 체크)
------------------------------

🔍 질문: 벤처확인 시스템 사업에서 '블록체인'이나 'AI', '메타버스' 기술을 도입하나요?
🎯 타겟: '벤처'
📖 [문서 1] (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .txt
   🛡️ [환각 방어 발동] 🚨 [치명적 경고] 질문에 포함된 키워드 '블록체인, AI, 메타버스'은(는) 이 문서에 단 한 번도 등장하지 않습니다. 절대 해당 기술이 포함되어 있다고 답하지 마세요. '관련 내용 없음'이라고 답하세요.

🎉 답변 생성 완료 (문서 1)
💡 Qwen 답변:
"문서에는 이러한 기술들이 언급되지 않았어요."

🧪 [Test 3] 📅 일정(정보 추출)
------------------------------

🔍 질문: 한영대학교 학사정보시스템 고도화 사업의 총 사업 기간은 몇 개월인가요?
🎯 타겟: '한영'
📖 [문서 1] 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.txt

🎉 답변 생성 완료 (문서 1)
💡 Qwen 답변:
총 사업 기간은 3개월이며, 여기에는 초기 준비 시간이 포함돼요.

🧪 [Test 4] 📝 요약(비정형 데이터)
---------------

In [18]:
# [Cell 7] 프롬프트 라우팅 기반 최종 RAG 시스템
import re
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# === 1. 프롬프트 템플릿 창고 (Templates) ===

# (A) 예산 전용: 계산 금지, 복사 붙여넣기 강제
budget_template = """<|im_start|>system
당신은 '숫자 확인 봇'입니다.
오직 [DB 힌트]에 있는 금액을 확인하여 정답을 말하세요.
[절대 규칙]
1. 예산 금액은 반드시 '원' 단위로 답하세요. (표기 금액은 달러($)가 아닌 원화입니다.)
2. [힌트]에 있는 금액은 $가 아닌 원화(￦) 단위입니다. 숫자를 임의로 바꾸거나 환율 계산을 금지합니다.
3. 숫자는 '352,000,000원' 처럼 한화 단위로 정확히 표기하세요. '$'는 쓰지 않습니다.
<|im_end|>
<|im_start|>user
[DB 힌트]
{hint}

[문서 내용]
{context}

[질문]
{question}
<|im_end|>
<|im_start|>assistant
"""

# (B) 요약 전용: 3줄 제한, 한국어 강제, 반복 차단
summary_template = """<|im_start|>system
당신은 유능한 요약 전문가입니다.
문서의 핵심 내용을 파악하여 요청사항에 맞게 요약하세요.
[절대 규칙]
1. 한국어로 명확하게 작성하세요. (영어 사용은 피하세요.)
2. 가장 중요한 내용을 추려서 번호 매기기(1., 2., 3.)로 나열하세요.
3. 3가지일 경우 작성이 끝나면 즉시 답변을 멈추세요. (반복하지 마세요)
<|im_end|>
<|im_start|>user
[문서 내용]
{context}

[질문]
{question}
<|im_end|>
<|im_start|>assistant
핵심 요약:
"""

# (C) 일반/검증 전용 (기존 로직 유지)
general_template = """<|im_start|>system
당신은 정직한 AI 컨설턴트입니다.
[검증 결과]를 절대적으로 신뢰하세요.
만약 [검증 결과]에서 "키워드가 없다"고 하면, 절대 거짓말하지 말고 "관련 내용 없음"이라고 답하세요.
<|im_end|>
<|im_start|>user
[검증 결과]
{hint}

[문서 내용]
{context}

[질문]
{question}
<|im_end|>
<|im_start|>assistant
"""

# === 2. 라우터 및 실행 함수 ===
def ask_qwen_final_router(query, target_file_keyword):
    print(f"\n🔍 질문: {query}")
    print(f"🎯 타겟: '{target_file_keyword}'")
    
    # 1. 문서 검색
    relevant_docs = retrieve_with_filter(query, target_file_keyword)
    if not relevant_docs:
        print("⚠️ 관련 문서를 찾지 못했습니다.")
        return

    found_answer = False
    
    for i, doc in enumerate(relevant_docs):
        filename = os.path.basename(doc.metadata['source'])
        file_stem = os.path.splitext(filename)[0]
        
        # 2. 힌트 및 검증 데이터 준비
        # (A) 예산 힌트 조회
        budget_hint = "정보 없음"
        if file_stem in budget_map: budget_hint = budget_map[file_stem]
        else:
            for key, val in budget_map.items():
                if key in file_stem or file_stem in key:
                    budget_hint = val; break
        
        # (B) 팩트 검증 (FactVerifier는 Cell 5에서 정의된 것 사용)
        fact_check_msg = verifier.verify(doc.page_content, query)

        # 3. ★ 프롬프트 라우팅 (핵심 로직) ★
        if "예산" in query or "금액" in query or "얼마" in query:
            print(f"   ⚙️ 모드 전환: [💰 예산 모드] (힌트: {budget_hint})")
            selected_template = budget_template
            final_hint = f"DB상의 정확한 예산은 '{budget_hint}'입니다."
            
        elif "요약" in query or "내용" in query or "정리" in query:
            print(f"   ⚙️ 모드 전환: [📝 요약 모드]")
            selected_template = summary_template
            final_hint = "" # 요약엔 힌트 필요 없음
            
        else:
            print(f"   ⚙️ 모드 전환: [🛡️ 일반/검증 모드]")
            selected_template = general_template
            final_hint = fact_check_msg
            if "치명적 경고" in fact_check_msg:
                print(f"   🛡️ [환각 방어] {fact_check_msg.strip()[:50]}...")

        # 4. 체인 실행
        prompt = PromptTemplate.from_template(selected_template)
        chain = prompt | llm | StrOutputParser()
        
        try:
            response = chain.invoke({
                "context": doc.page_content, 
                "question": query,
                "hint": final_hint
            })
            
            cleaned_response = response.replace("<|im_end|>", "").strip()

            # "PASS" 필터링 (일반 모드일 때만)
            if "PASS" not in cleaned_response and len(cleaned_response) > 2:
                print(f"\n🎉 답변 생성 완료 (문서 {i+1})")
                print("=" * 50)
                print(f"💡 Qwen 답변:\n{cleaned_response}")
                print("=" * 50)
                found_answer = True
                break
                
        except Exception as e:
            print(f"⚠️ 에러: {e}")
            
    if not found_answer:
        print("\n💨 실패.")

# === 3. 재검증 테스트 ===
print("🚀 [최종 수정] 라우터 시스템 테스트 시작")

# (1) 아까 실패했던 예산 (35억 오류 수정 확인)
ask_qwen_final_router(
    "벤처확인종합관리시스템 기능 고도화 용역사업의 예산은 얼마인가요?", 
    "벤처"
)

# (2) 아까 폭주했던 요약 (무한 반복 수정 확인)
ask_qwen_final_router(
    "국민연금공단 이러닝 사업의 주요 과업 내용 3가지만 요약해 주세요.", 
    "국민연금"
)

🚀 [최종 수정] 라우터 시스템 테스트 시작

🔍 질문: 벤처확인종합관리시스템 기능 고도화 용역사업의 예산은 얼마인가요?
🎯 타겟: '벤처'
   ⚙️ 모드 전환: [💰 예산 모드] (힌트: 352,000,000원)

🎉 답변 생성 완료 (문서 1)
💡 Qwen 답변:
352,000,000 won이에요.

🔍 질문: 국민연금공단 이러닝 사업의 주요 과업 내용 3가지만 요약해 주세요.
🎯 타겟: '국민연금'
   ⚙️ 모드 전환: [📝 요약 모드]

🎉 답변 생성 완료 (문서 1)
💡 Qwen 답변:
1. **Training Road Map**: 국민연금공단(NPC)의 리더십, 직무 또는 일반 역량을 목표로 하는 일련의 교육 과정을 만들어요.
2. **Continuous Education Programs**: 특정 수료 기준을 충족하거나 추가 검토가 필요한 과정을 진행하기 위해 수료 점수가 결정돼요.
3. **Course Content and Delivery Methods**: 각 과정의 난이도와 형식을 설정하며, 여기에는 수강 시간 제한 및 수료 여부를 판단하기 위한 평가가 포함되어 있어요.

이것들이 NPC의 교육 과정 운영에 대한 주요 작업이며, 더 많은 세부 사항이나 예시는 [별록]을 참고해주세요.
